# SHL Hiring Assessment 2026 — Grammar Scoring Engine

## Objective
Build a regression model that predicts a continuous spoken-grammar score from **0 to 5** using both the speech transcript and acoustic characteristics of the audio.

## Final approach
**WAV audio → Whisper transcription → handcrafted text features + acoustic features → Random Forest + Gradient Boosting → 50/50 ensemble → test predictions**

The final multimodal representation contains **41 features: 8 text + 33 audio features**.

## Evaluation
The competition uses **RMSE** and **Pearson correlation**. Because the test labels are hidden, model selection is based on **5-fold out-of-fold validation** on the training set. The final notebook also reports the required training RMSE.


In [ ]:
# Set reproducibility and Kaggle paths
import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_PATH = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"
TRAIN_AUDIO_PATH = f"{BASE_PATH}/train"
TEST_AUDIO_PATH = f"{BASE_PATH}/test"

TRAIN_CSV = f"{BASE_PATH}/train.csv"
TEST_CSV = f"{BASE_PATH}/test.csv"
SAMPLE_CSV = f"{BASE_PATH}/sample_submission.csv"


In [ ]:
# Load the competition datasets
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
sample_df = pd.read_csv(SAMPLE_CSV)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_df.shape)
print("Train columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())


In [ ]:
# Validate labels, filenames, and audio files
train_files = set(os.listdir(TRAIN_AUDIO_PATH))
test_files = set(os.listdir(TEST_AUDIO_PATH))

print("Label range:", train_df["label"].min(), "to", train_df["label"].max())
print("Unique labels:", sorted(train_df["label"].unique()))
print("Missing train values:", train_df.isna().sum().sum())
print("Missing test values:", test_df.isna().sum().sum())
print("Duplicate train filenames:", train_df["filename"].duplicated().sum())
print("Duplicate test filenames:", test_df["filename"].duplicated().sum())
print("Missing train audio files:", len(set(train_df["filename"]) - train_files))
print("Missing test audio files:", len(set(test_df["filename"]) - test_files))
print("Sample submission rows do not match test rows:", len(sample_df) != len(test_df))


## 1. Data overview and label distribution

The training set contains **769 labeled audio samples** and the test set contains **216 unlabeled samples**. The training labels range from **0 to 5**.


In [ ]:
# Plot the training grammar-score distribution
label_counts = train_df["label"].value_counts().sort_index()

plt.figure(figsize=(8, 5))
plt.bar(label_counts.index.astype(str), label_counts.values)
plt.xlabel("Grammar score")
plt.ylabel("Number of samples")
plt.title("Training Grammar Score Distribution")
plt.tight_layout()
plt.show()


## 2. Audio inspection

The audio files are loaded at their original sample rate for inspection. Feature extraction later uses a consistent **16 kHz** sampling rate.


In [ ]:
# Inspect one training audio file
import librosa

sample_filename = train_df.iloc[0]["filename"]
sample_audio_path = os.path.join(TRAIN_AUDIO_PATH, sample_filename)
y_audio, sr = librosa.load(sample_audio_path, sr=None)

print("Filename:", sample_filename)
print("Label:", train_df.iloc[0]["label"])
print("Sample rate:", sr)
print("Duration:", round(len(y_audio) / sr, 2), "seconds")
print("RMS:", round(float(np.sqrt(np.mean(y_audio ** 2))), 5))


## 3. Whisper transcription

Whisper **base** is used to convert each spoken response into text. A cache is used when the corrected transcript file already exists; otherwise the notebook transcribes the training audio. Any missing transcripts are automatically retried before feature extraction.


In [ ]:
# Install Whisper when running in a fresh Kaggle session
!pip install -q openai-whisper


In [ ]:
# Load the Whisper speech-to-text model
import whisper
from tqdm.auto import tqdm

whisper_model = whisper.load_model("base")
print("Whisper loaded successfully.")


In [ ]:
# Transcribe and cache all training audio
train_cache = "/kaggle/working/train_with_transcripts.csv"

if os.path.exists(train_cache):
    cached_train = pd.read_csv(train_cache)
    cache_matches = (
        len(cached_train) == len(train_df)
        and cached_train["filename"].tolist() == train_df["filename"].tolist()
        and "transcript" in cached_train.columns
        and cached_train["transcript"].notna().all()
    )
else:
    cache_matches = False

if cache_matches:
    train_df = cached_train.copy()
    print("Loaded corrected training transcripts from cache.")
else:
    transcripts = []
    for filename in tqdm(train_df["filename"], desc="Transcribing training audio"):
        audio_path = os.path.join(TRAIN_AUDIO_PATH, filename)
        result = whisper_model.transcribe(audio_path, fp16=True, language="en")
        transcripts.append(result["text"].strip())
    train_df["transcript"] = transcripts

    missing_idx = train_df.index[train_df["transcript"].isna() | (train_df["transcript"].str.strip() == "")].tolist()
    for idx in missing_idx:
        filename = train_df.loc[idx, "filename"]
        audio_path = os.path.join(TRAIN_AUDIO_PATH, filename)
        result = whisper_model.transcribe(audio_path, fp16=True, language="en")
        train_df.loc[idx, "transcript"] = result["text"].strip()

    train_df.to_csv(train_cache, index=False)

print("Training transcripts:", len(train_df))
print("Missing transcripts:", train_df["transcript"].isna().sum())
print("Sample transcript:\n", train_df.iloc[0]["transcript"])


## 4. Text feature engineering

Eight lightweight features summarize transcript complexity and hesitation:

- word count
- sentence count
- average sentence length
- average word length
- vocabulary diversity
- repeated-word count
- repeated-word ratio
- filler-word count


In [ ]:
# Define the handcrafted transcript feature extractor
def extract_text_features(text):
    if not isinstance(text, str):
        text = ""

    words = re.findall(r"\b[a-zA-Z]+\b", text.lower())
    word_count = len(words)
    unique_word_count = len(set(words))

    sentences = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]
    sentence_count = len(sentences)

    avg_sentence_length = word_count / sentence_count if sentence_count else 0
    avg_word_length = np.mean([len(w) for w in words]) if words else 0
    vocabulary_diversity = unique_word_count / word_count if word_count else 0
    repeated_word_count = word_count - unique_word_count
    repeated_word_ratio = repeated_word_count / word_count if word_count else 0

    filler_words = {"um", "uh", "erm", "hmm", "like", "you", "know"}
    filler_count = sum(word in filler_words for word in words)

    return {
        "word_count": word_count,
        "sentence_count": sentence_count,
        "avg_sentence_length": avg_sentence_length,
        "avg_word_length": avg_word_length,
        "vocabulary_diversity": vocabulary_diversity,
        "repeated_word_count": repeated_word_count,
        "repeated_word_ratio": repeated_word_ratio,
        "filler_count": filler_count,
    }


In [ ]:
# Build the corrected training text features
feature_df = train_df["transcript"].apply(extract_text_features).apply(pd.Series)

print("Text feature shape:", feature_df.shape)
print("Missing text features:", feature_df.isna().sum().sum())
display(feature_df.head())


## 5. Audio feature engineering

The acoustic representation contains **33 features**: duration, RMS energy statistics, zero-crossing-rate statistics, pitch statistics, and mean/std values for 13 MFCCs.


In [ ]:
# Define the compact acoustic feature extractor
def extract_audio_features(audio_path):
    y_audio, sr = librosa.load(audio_path, sr=16000)

    duration = len(y_audio) / sr
    rms = librosa.feature.rms(y=y_audio)[0]
    zcr = librosa.feature.zero_crossing_rate(y_audio)[0]

    f0 = librosa.yin(y_audio, fmin=70, fmax=400, sr=sr)
    f0_valid = f0[np.isfinite(f0)]

    mfcc = librosa.feature.mfcc(y=y_audio, sr=sr, n_mfcc=13)

    return {
        "audio_duration": duration,
        "rms_mean": np.mean(rms),
        "rms_std": np.std(rms),
        "zcr_mean": np.mean(zcr),
        "zcr_std": np.std(zcr),
        "pitch_mean": np.mean(f0_valid) if len(f0_valid) else 0,
        "pitch_std": np.std(f0_valid) if len(f0_valid) else 0,
        **{f"mfcc_{i+1}_mean": np.mean(mfcc[i]) for i in range(13)},
        **{f"mfcc_{i+1}_std": np.std(mfcc[i]) for i in range(13)},
    }


In [ ]:
# Extract acoustic features from every training audio file
audio_features = []

for filename in tqdm(train_df["filename"], desc="Extracting training audio features"):
    audio_features.append(
        extract_audio_features(os.path.join(TRAIN_AUDIO_PATH, filename))
    )

audio_feature_df = pd.DataFrame(audio_features)
combined_audio_df = pd.concat([feature_df, audio_feature_df], axis=1)

print("Audio feature shape:", audio_feature_df.shape)
print("Combined feature shape:", combined_audio_df.shape)
print("Missing combined features:", combined_audio_df.isna().sum().sum())


## 6. Model development and validation

A simple mean-prediction baseline is followed by multimodal tree models. The final model family was selected using **5-fold cross-validation** and out-of-fold predictions.


In [ ]:
# Establish a mean-prediction validation baseline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

X_text = feature_df
y = train_df["label"].values


In [ ]:
# Evaluate the text-only Random Forest baseline
from sklearn.ensemble import RandomForestRegressor

X_train, X_val, y_train, y_val = train_test_split(
    feature_df, train_df["label"], test_size=0.2, random_state=SEED
)

text_rf = RandomForestRegressor(
    n_estimators=300, max_depth=6, min_samples_leaf=3,
    random_state=SEED, n_jobs=-1
)
text_rf.fit(X_train, y_train)
text_pred = text_rf.predict(X_val)

print("Text-only RMSE:", round(np.sqrt(mean_squared_error(y_val, text_pred)), 4))
print("Text-only Pearson:", round(pearsonr(y_val, text_pred)[0], 4))


In [ ]:
# Compare the multimodal Random Forest with 5-fold cross-validation
from sklearn.model_selection import KFold, cross_val_score

X = combined_audio_df.values
y = train_df["label"].values
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

rf_base = RandomForestRegressor(
    n_estimators=300, max_depth=6, min_samples_leaf=3,
    random_state=SEED, n_jobs=-1
)

rf_scores = -cross_val_score(
    rf_base, X, y, cv=kf,
    scoring="neg_root_mean_squared_error", n_jobs=-1
)

print("Fold RMSE:", np.round(rf_scores, 4))
print("Mean RMSE:", round(rf_scores.mean(), 4))
print("Std RMSE:", round(rf_scores.std(), 4))


In [ ]:
# Generate multimodal Random Forest out-of-fold predictions
rf_oof_base = np.zeros(len(y))

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    model = RandomForestRegressor(
        n_estimators=300, max_depth=6, min_samples_leaf=3,
        random_state=SEED, n_jobs=-1
    )
    model.fit(X[train_idx], y[train_idx])
    rf_oof_base[val_idx] = model.predict(X[val_idx])

base_oof_rmse = np.sqrt(mean_squared_error(y, rf_oof_base))
base_oof_pearson = pearsonr(y, rf_oof_base)[0]

print("Base multimodal OOF RMSE:", round(base_oof_rmse, 4))
print("Base multimodal OOF Pearson:", round(base_oof_pearson, 4))


## 7. Hyperparameter tuning and ensemble selection

The tuned Random Forest uses **300 trees, max depth 10, minimum leaf size 2, and all features per split**. A Huber-loss Gradient Boosting model is evaluated alongside it. Their 50/50 out-of-fold ensemble was the best validated configuration in the experiments recorded in this notebook.


In [ ]:
# Evaluate the tuned Random Forest with 5-fold cross-validation
rf_tuned_template = RandomForestRegressor(
    n_estimators=300, max_depth=10, min_samples_leaf=2,
    max_features=1.0, random_state=SEED, n_jobs=-1
)

rf_tuned_scores = -cross_val_score(
    rf_tuned_template, X, y, cv=kf,
    scoring="neg_root_mean_squared_error", n_jobs=-1
)

print("Tuned RF fold RMSE:", np.round(rf_tuned_scores, 4))
print("Tuned RF mean RMSE:", round(rf_tuned_scores.mean(), 4))
print("Tuned RF std RMSE:", round(rf_tuned_scores.std(), 4))


In [ ]:
# Generate tuned RF and Gradient Boosting out-of-fold predictions
from sklearn.ensemble import GradientBoostingRegressor

rf_oof = np.zeros(len(y))
gb_oof = np.zeros(len(y))

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    rf_model = RandomForestRegressor(
        n_estimators=300, max_depth=10, min_samples_leaf=2,
        max_features=1.0, random_state=SEED, n_jobs=-1
    )
    gb_model = GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.03, max_depth=2,
        min_samples_leaf=5, loss="huber", random_state=SEED
    )

    rf_model.fit(X[train_idx], y[train_idx])
    gb_model.fit(X[train_idx], y[train_idx])

    rf_oof[val_idx] = rf_model.predict(X[val_idx])
    gb_oof[val_idx] = gb_model.predict(X[val_idx])

rf_oof_rmse = np.sqrt(mean_squared_error(y, rf_oof))
gb_oof_rmse = np.sqrt(mean_squared_error(y, gb_oof))
ensemble_oof = 0.5 * rf_oof + 0.5 * gb_oof
ensemble_oof_rmse = np.sqrt(mean_squared_error(y, ensemble_oof))
ensemble_oof_pearson = pearsonr(y, ensemble_oof)[0]

print("Tuned RF OOF RMSE:", round(rf_oof_rmse, 4))
print("Gradient Boosting OOF RMSE:", round(gb_oof_rmse, 4))
print("50/50 Ensemble OOF RMSE:", round(ensemble_oof_rmse, 4))
print("50/50 Ensemble OOF Pearson:", round(ensemble_oof_pearson, 4))


In [ ]:
# Visualize validation performance across model variants
model_names = ["Text RF", "Base multimodal RF", "Tuned RF", "Gradient Boosting", "50/50 Ensemble"]
model_rmse = [1.0409, 0.7772, 0.7590, 0.7587, 0.7513]

plt.figure(figsize=(9, 5))
plt.bar(model_names, model_rmse)
plt.ylabel("OOF / validation RMSE")
plt.title("Model Performance Comparison")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Visualize actual versus out-of-fold ensemble predictions
plt.figure(figsize=(7, 6))
plt.scatter(y, ensemble_oof, alpha=0.6)
plt.plot([0, 5], [0, 5], linestyle="--")
plt.xlabel("Actual grammar score")
plt.ylabel("OOF predicted score")
plt.title("Actual vs Out-of-Fold Ensemble Predictions")
plt.xlim(0, 5)
plt.ylim(0, 5)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
# Visualize absolute error by actual grammar score
residual_df = pd.DataFrame({
    "actual": y,
    "predicted": ensemble_oof
})
residual_df["absolute_error"] = np.abs(residual_df["predicted"] - residual_df["actual"])
score_error = residual_df.groupby("actual")["absolute_error"].mean().reset_index()

plt.figure(figsize=(8, 5))
plt.bar(score_error["actual"].astype(str), score_error["absolute_error"])
plt.xlabel("Actual grammar score")
plt.ylabel("Mean absolute error")
plt.title("Residual Error by Grammar Score")
plt.tight_layout()
plt.show()


In [ ]:
# Visualize the most important multimodal features
rf_importance_model = RandomForestRegressor(
    n_estimators=300, max_depth=10, min_samples_leaf=2,
    max_features=1.0, random_state=SEED, n_jobs=-1
)
rf_importance_model.fit(X, y)

importance_df = pd.DataFrame({
    "feature": combined_audio_df.columns,
    "importance": rf_importance_model.feature_importances_
}).sort_values("importance", ascending=False).head(15)

plt.figure(figsize=(9, 6))
plt.barh(importance_df["feature"].iloc[::-1], importance_df["importance"].iloc[::-1])
plt.xlabel("Feature importance")
plt.title("Top 15 Random Forest Features")
plt.tight_layout()
plt.show()


## 8. Final model and required training evaluation

The final ensemble is trained on **all 769 labeled samples**. The training RMSE is reported because it is compulsory for the competition notebook; the OOF RMSE remains the more meaningful generalization estimate.


In [ ]:
# Train the final Random Forest and Gradient Boosting models
rf_final = RandomForestRegressor(
    n_estimators=300, max_depth=10, min_samples_leaf=2,
    max_features=1.0, random_state=SEED, n_jobs=-1
)

gb_final = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.03, max_depth=2,
    min_samples_leaf=5, loss="huber", random_state=SEED
)

rf_final.fit(combined_audio_df, y)
gb_final.fit(combined_audio_df, y)
print("Final models trained successfully.")


In [ ]:
# Calculate the final ensemble training RMSE and Pearson correlation
rf_train_pred = rf_final.predict(combined_audio_df)
gb_train_pred = gb_final.predict(combined_audio_df)
ensemble_train_pred = 0.5 * rf_train_pred + 0.5 * gb_train_pred

final_train_rmse = np.sqrt(mean_squared_error(y, ensemble_train_pred))
final_train_pearson = pearsonr(y, ensemble_train_pred)[0]

print(f"Final Ensemble Training RMSE: {final_train_rmse:.4f}")
print(f"Final Ensemble Training Pearson: {final_train_pearson:.4f}")


## 9. Test transcription and feature extraction

The same preprocessing used for training is applied to all **216 test audio files**. No labels from the test set are used.


In [ ]:
# Transcribe and cache all test audio
TEST_CACHE = "/kaggle/working/test_with_transcripts.csv"

if os.path.exists(TEST_CACHE):
    cached_test = pd.read_csv(TEST_CACHE)
    cache_matches = (
        len(cached_test) == len(test_df)
        and cached_test["filename"].tolist() == test_df["filename"].tolist()
        and "transcript" in cached_test.columns
        and cached_test["transcript"].notna().all()
    )
else:
    cache_matches = False

if cache_matches:
    test_df = cached_test.copy()
    print("Loaded test transcripts from cache.")
else:
    test_transcripts = []
    for filename in tqdm(test_df["filename"], desc="Transcribing test audio"):
        audio_path = os.path.join(TEST_AUDIO_PATH, filename)
        result = whisper_model.transcribe(audio_path, fp16=True, language="en")
        test_transcripts.append(result["text"].strip())
    test_df["transcript"] = test_transcripts
    test_df.to_csv(TEST_CACHE, index=False)

print("Test transcripts:", len(test_df))
print("Missing test transcripts:", test_df["transcript"].isna().sum())


In [ ]:
# Build the 41-dimensional test feature matrix
test_feature_df = test_df["transcript"].apply(extract_text_features).apply(pd.Series)

print("Test text feature shape:", test_feature_df.shape)
print("Missing test text features:", test_feature_df.isna().sum().sum())


In [ ]:
# Extract test audio features
test_audio_features = []

for filename in tqdm(test_df["filename"], desc="Extracting test audio features"):
    test_audio_features.append(
        extract_audio_features(os.path.join(TEST_AUDIO_PATH, filename))
    )

test_audio_df = pd.DataFrame(test_audio_features)
test_combined_df = pd.concat([test_feature_df, test_audio_df], axis=1)

print("Test audio feature shape:", test_audio_df.shape)
print("Test combined feature shape:", test_combined_df.shape)
print("Missing test features:", test_combined_df.isna().sum().sum())
print("Training/test columns match:", list(combined_audio_df.columns) == list(test_combined_df.columns))


## 10. Generate predictions and submission


In [ ]:
# Generate the final 50/50 ensemble test predictions
rf_test_pred = rf_final.predict(test_combined_df)
gb_test_pred = gb_final.predict(test_combined_df)
test_predictions = np.clip(0.5 * rf_test_pred + 0.5 * gb_test_pred, 0, 5)

print("Test predictions:", len(test_predictions))
print("Prediction range:", test_predictions.min(), "to", test_predictions.max())


In [ ]:
# Create and validate the final Kaggle submission
submission = pd.DataFrame({
    "filename": test_df["filename"],
    "label": test_predictions
})

submission_path = "/kaggle/working/submission.csv"
submission.to_csv(submission_path, index=False)

print("Submission shape:", submission.shape)
print("Duplicate filenames:", submission["filename"].duplicated().sum())
print("Missing filenames:", submission["filename"].isna().sum())
print("Missing labels:", submission["label"].isna().sum())
print("Labels below 0:", (submission["label"] < 0).sum())
print("Labels above 5:", (submission["label"] > 5).sum())
print("Columns:", submission.columns.tolist())
display(submission.head(10))


## 11. Final results summary

| Metric | Result |
|---|---:|
| Training samples | 769 |
| Test samples | 216 |
| Final features | 41 (8 text + 33 audio) |
| Tuned RF OOF RMSE | 0.7590 |
| Gradient Boosting CV RMSE | 0.7587 |
| **50/50 Ensemble OOF RMSE** | **0.7513** |
| **50/50 Ensemble OOF Pearson** | **0.7965** |
| **Final Ensemble Training RMSE** | **0.4764** |
| Final Ensemble Training Pearson | 0.9320 |

### Conclusion
The multimodal representation consistently outperformed the text-only approach. Combining transcript-derived linguistic features with acoustic characteristics produced the strongest validated result, and the final 50/50 Random Forest + Gradient Boosting ensemble was used to generate the 216 test predictions.
